# Method 2 — MTZ formulation (Gurobi)

The **Miller–Tucker–Zemlin** model keeps the same arc variables but replaces the
exponential family of subset cuts with a polynomial number of order variables $u_i$.

$$
\begin{aligned}
\min\quad & \sum_{i}\sum_{j\ne i} d_{ij}x_{ij} \\
\text{s.t.}\quad & \sum_{j\ne i}x_{ij}=1,\quad \sum_{j\ne i}x_{ji}=1 && \forall i \\
& u_i-u_j+n x_{ij}\le n-1 && \forall i,j\in\{1,\dots,n-1\},\ i\ne j \\
& u_0=1,\quad 1\le u_i\le n-1 && \\
& x_{ij}\in\{0,1\}
\end{aligned}
$$

The order variables make a subtour impossible: each visited city must have a strictly
larger $u$ value than the previous city, but the values cannot increase forever inside a
cycle.

The MTZ model is much smaller — about $n^2+n$ variables for 14 cities, instead of 8,192
subset constraints — but its LP relaxation is weaker than DFJ.

## 1. Load the data

In [ ]:
from tsp_solver import (
    DEFAULT_DATA, load_cities, distance_matrix,
    solve_mtz, held_karp, GurobiUnavailable,
    plot_route, route_names,
)

cities = load_cities(DEFAULT_DATA)
distances = distance_matrix(cities)
cities

## 2. Solve with the MTZ model

In [ ]:
try:
    result = solve_mtz(distances, time_limit=120, output_flag=False)
except GurobiUnavailable as error:
    print(f"[fallback] {error}")
    result = held_karp(distances)

print(result.method)
print(f"cost: {result.cost:.2f} km")
print("tour:", " -> ".join(route_names(result.tour, cities)))

## 3. Plot the route

In [ ]:
plot_route(cities, result.tour, result.cost, "results/mtz_route.png", "MTZ route")
from IPython.display import Image
Image(filename="results/mtz_route.png")

## DFJ vs MTZ

| | DFJ | MTZ |
|---|---|---|
| Subtour elimination | All subset constraints | Order variables + big-M |
| Constraints | Exponential | Polynomial |
| LP relaxation | Tight | Weaker |
| Best for | Small exact instances | Larger models, warm starts |

Both formulations should return the same optimal objective on a symmetric instance.
Run `python tsp_solver.py --gurobi` to compare them and the heuristic in one table.